In [3]:
from PIL import Image
import glob
import os

from mpmath.identification import transforms
from torch.utils.data import DataLoader, Dataset


class CelebADataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.paths = sorted(glob.glob(os.path.join(root, '*.jpg')))

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img



In [2]:
import torch.nn as nn
import torch
import torch.nn.functional as F


class ConvVAE(nn.Module):
    def __init__(self, latent_dim=128, ch=64, image_size=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.ch = ch
        self.image_size = image_size

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, ch, 4, 2, 1),
            nn.ReLU(True),
            nn.Conv2d(ch, ch * 2, 4, 2, 1),
            nn.BatchNorm2d(ch * 2),
            nn.ReLU(True),
            nn.Conv2d(ch * 2, ch * 4, 4, 2, 1),
            nn.BatchNorm2d(ch * 4),
            nn.ReLU(True),
            nn.Conv2d(ch * 4, ch * 8, 4, 2, 1),
            nn.BatchNorm2d(ch * 8),
            nn.ReLU(True),
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(ch * 8, ch * 4, 4, 2, 1),
            nn.BatchNorm2d(ch * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(ch * 4, ch * 2, 4, 2, 1),
            nn.BatchNorm2d(ch * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d(ch * 2, ch, 4, 2, 1),
            nn.BatchNorm2d(ch),
            nn.ReLU(True),
            nn.ConvTranspose2d(ch, 3, 4, 2, 1),
            nn.Sigmoid()
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 3, image_size, image_size)
            feat_dim = self.encoder(dummy).view(1, -1).size(1)

        self.fc_mu = nn.Linear(feat_dim, latent_dim)
        self.fc_logvar = nn.Linear(feat_dim, latent_dim)
        self.fc_dec = nn.Linear(latent_dim, feat_dim)
        self._feat_shape = self.encoder(dummy).shape[1:]

    def encode(self, x):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = self.fc_dec(z)
        h = h.view(h.size(0), *self._feat_shape)
        x_recon = self.decoder(h)
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

    def vae_loss(self, recon_x, x, mu, logvar):
        recon_loss = F.mse_loss(recon_x, x, reduction='sum')
        kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + kld, recon_loss, kld

In [ ]:
batch_size = 1024
latent_dim = 128
image_size = 64
num_epochs = 300

transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.CenterCrop(image_size),
    transforms.ToTensor(),
])


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

dataset = CelebADataset(root="/home/jammy/code/learning_py/deeplearning/data/celeb/img_align_celeb",
                        transform=transforms.ToTensor())
loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)


def train():
    model = ConvVAE(latent_dim=latent_dim, image_size=image_size).to(device)
    optim = torch.optim.Adam(model.parameters(), lr=0.001)
    global_step = 0
    for epoch in range(1, num_epochs + 1):
        model.train()
        epoch_loss = 0.0
        epoch_recon = 0.0
        epoch_kld = 0.0

        for batch_idx, imgs in enumerate(loader):
            imgs = imgs.to(device, non_blocking=True)
            optim.zero_grad()
            recon_imgs, mu, logvar = model(imgs)
            loss, recon_l, kld = model.vae_loss(recon_imgs, imgs, mu, logvar)
            loss.backward()
            optim.step()

            epoch_loss += loss.item()
            epoch_recon += recon_l.item()
            epoch_kld += kld.item()
            global_step += 1

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}/{num_epochs}, Step {global_step}/{len(loader)}, "
                      f"Loss {loss.item():.4f}"
                      f"recon {recon_l.item():.4f} kld {kld.item():.4f}"
                      )

